Hands-on lab: Snowflake Cortex + Gemini Enterprise — Iceberg, semantic views, and Cortex Agents
*Co-authored with CoCo*

# AI-Ready Open Lakehouse: Snowflake Cortex + Gemini Enterprise


**Duration:** ~75 min | **Level:** Intermediate

Build a governed data product — land open data on Iceberg, add a semantic layer and Cortex Agent, then consume the **same** agent from Snowflake CoWork **and** Google Gemini Enterprise over MCP.

> One copy of data. Many surfaces. Build the agent once, use it everywhere.

## Architecture Overview

The diagram below shows what we are building end-to-end. Data flows left-to-right: from the Snowflake Marketplace through Iceberg storage on GCS, up through a Semantic View and Cortex Agent, and out to two consumption surfaces — CoWork (native) and Gemini Enterprise (via MCP).

## Account Sectup

### Snowflake Account

- Go to dataops and register for a temporary Snowflake account for this lab using your email.
- Log in with the provided username and password to the given snowflake account
- this is snowflake UI called Snowsight, explore the left hand side main menu.

### GCP Account
@bsandell please complete

### Looker Account
@bsandell please complete


## Workspace
- Select workspaces from left pannel
- click + the very top to create a new workspace
- select from github and then use public for https://github.com/sfc-gh-akhosro/gcp-snowflake-solutions.git
> In practice, you will often use through personal token to pull and push.
- click on Add new (you can create python, sql, notebook, markdown, ... files in the workspace), cancel it, no action needed
- click on the file hol-cortex-gemini-ipynb 

## RBAC (Role Based Access Control)

We create two roles:
- **`hol_role`** — owns everything (database, warehouse, integrations, agents). This is the builder role.
- **`end_user_role`** — consumer role that can only *use* the agent through CoWork and Gemini Enterprise (MCP). This simulates a business user.

Both roles are granted to whoever is running this notebook (`CURRENT_USER()`).

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

-- Builder role: owns all workshop objects
CREATE ROLE IF NOT EXISTS hol_role;

-- Consumer role: can only use the agent (CoWork, Gemini Enterprise)
CREATE ROLE IF NOT EXISTS end_user_role;

-- Grant both roles to whoever is running this notebook
BEGIN
  LET usr := CURRENT_USER();
  EXECUTE IMMEDIATE 'GRANT ROLE hol_role TO USER ' || :usr;
  EXECUTE IMMEDIATE 'GRANT ROLE end_user_role TO USER ' || :usr;
END;

-- Builder privileges
GRANT CREATE DATABASE        ON ACCOUNT TO ROLE hol_role;
GRANT CREATE WAREHOUSE       ON ACCOUNT TO ROLE hol_role;
GRANT CREATE INTEGRATION     ON ACCOUNT TO ROLE hol_role;
GRANT CREATE EXTERNAL VOLUME ON ACCOUNT TO ROLE hol_role;

-- Cortex access for both roles
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE hol_role;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE end_user_role;

-- Switch to hol_role to build
USE ROLE hol_role;

-- Create owned resources
CREATE WAREHOUSE IF NOT EXISTS hol_wh
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE hol_wh;

CREATE DATABASE IF NOT EXISTS hol_db;
USE SCHEMA hol_db.public;

-- Schema-level privileges (must come after database/schema exist)
USE ROLE ACCOUNTADMIN;
GRANT CREATE SEMANTIC VIEW ON SCHEMA hol_db.public TO ROLE hol_role;
USE ROLE hol_role;
USE WAREHOUSE hol_wh;
USE SCHEMA hol_db.public;

-- Grant consumer role usage on warehouse and database
GRANT USAGE ON WAREHOUSE hol_wh TO ROLE end_user_role;
GRANT USAGE ON DATABASE hol_db TO ROLE end_user_role;
GRANT USAGE ON SCHEMA hol_db.public TO ROLE end_user_role;

-- Verify context
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS wh, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

## Iceberg Bucket

We need a GCS bucket that Snowflake can write Iceberg data to. In the workshop everyone uses: `gs://hands-on-lab-cortex-gemini`

**Steps in Google Cloud Console:**

1. Go to **Cloud Storage → Buckets** → verify `hands-on-lab-cortex-gemini` exists (or create it — `us-central1` or multi-region `US` both work)
2. Run the SQL cell below — it creates the external volume and extracts the GCS service account from the JSON output
3. Copy the **service account email** from the query result
4. Back in GCS Console → bucket `hands-on-lab-cortex-gemini` → **Permissions** → **Grant Access**
   - **New principal:** paste the service account email
   - **Role:** select **Storage Admin**
   - Click **Save**

> Without this permission grant, Snowflake cannot write Iceberg metadata/data files to the bucket and the next step will fail.

In [ ]:
-- Create an external volume pointing to the shared GCS bucket
CREATE OR REPLACE EXTERNAL VOLUME hol_gcs_vol
  STORAGE_LOCATIONS = ((
    NAME = 'hol-gcs'
    STORAGE_PROVIDER = 'GCS'
    STORAGE_BASE_URL = 'gcs://hands-on-lab-cortex-gemini/iceberg/'
  ));

-- Describe to get storage config, capture query ID immediately
DESCRIBE EXTERNAL VOLUME hol_gcs_vol;
SET desc_qid = LAST_QUERY_ID();

-- Extract the GCS service account using Snowflake JSON parsing
-- PARSE_JSON converts the stored JSON string → dot notation pulls the exact field
SELECT
  PARSE_JSON("property_value"):STORAGE_GCP_SERVICE_ACCOUNT::STRING
    AS gcs_service_account_to_grant
FROM TABLE(RESULT_SCAN($desc_qid))
WHERE "property" = 'STORAGE_LOCATION_1';

## Snowflake Marketplace

Snowflake Marketplace lets you discover and access shared datasets with a click — no ETL, no copies, instant access.

1. In Snowsight, go to **Data Products → Marketplace**
2. Search for **"Snowflake Public Data"** → click **Get** on the **Free** listing
3. Name the database `SNOWFLAKE_PUBLIC_DATA_FREE` → click **Get**

> There is no SQL command to install a marketplace listing — the UI is required. Under the hood, Snowflake creates a read-only database from the provider's share.

We'll use four tables from this dataset:
- **BLS Price Timeseries** — CPI index (inflation)
- **BLS Employment Timeseries** — unemployment rate by state
- **Freddie Mac Housing Timeseries** — 30-year & 15-year mortgage rates
- **IRS Individual Income Timeseries** — average income per tax return by state

Run the cell below to confirm access.

In [ ]:
-- Verify access to all four source tables
SELECT 'BLS_PRICE' AS source, COUNT(*) AS row_count FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
UNION ALL
SELECT 'BLS_EMPLOYMENT', COUNT(*) FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
UNION ALL
SELECT 'FREDDIE_MAC', COUNT(*) FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
UNION ALL
SELECT 'IRS_INCOME', COUNT(*) FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES;

## Iceberg Tables

We join the three marketplace sources into a single **wide-format** Iceberg table on your GCS bucket — one row per month with all key economic metrics side-by-side. This is a proper analytics-ready fact table in open Iceberg format.

In [ ]:
-- Create Iceberg table: national metrics + state-level unemployment & income
CREATE OR REPLACE ICEBERG TABLE hol_db.public.economic_indicators
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'hol_gcs_vol'
  BASE_LOCATION = 'economic_indicators'
  AS
WITH cpi AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    AVG(value) AS cpi_index
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
  WHERE variable_name = 'CPI: All items, Not seasonally adjusted, Monthly'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
mortgage_30yr AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    ROUND(AVG(value) * 100, 2) AS mortgage_rate_30yr_pct
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
  WHERE variable_name = '30-Year Fixed Rate Mortgage Rate, National Average'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
mortgage_15yr AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    ROUND(AVG(value) * 100, 2) AS mortgage_rate_15yr_pct
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
  WHERE variable_name = '15-Year Fixed Rate Mortgage Rate, National Average'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
unemployment AS (
  SELECT
    DATE_TRUNC('month', date) AS month,
    geo_id,
    AVG(value) AS unemployment_rate_pct
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
  WHERE variable_name = 'Local Area Unemployment: Unemployment Rate, Not seasonally adjusted, Monthly'
    AND LENGTH(geo_id) = 8
  GROUP BY 1, 2
),
national_unemployment AS (
  SELECT month, ROUND(AVG(unemployment_rate_pct), 2) AS unemployment_rate_pct
  FROM unemployment
  GROUP BY 1
),
-- IRS: average income per return by state (annual)
income_raw AS (
  SELECT
    agi.geo_id,
    YEAR(agi.date) AS yr,
    ROUND(agi.value / NULLIF(ret.value, 0), 0) AS avg_income_per_return
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES agi
  JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES ret
    ON agi.geo_id = ret.geo_id AND agi.date = ret.date
  WHERE agi.variable_name = 'Adjusted gross income (AGI), AGI bin: Total'
    AND ret.variable_name = 'Number of returns, AGI bin: Total'
    AND LENGTH(agi.geo_id) = 8
),
-- Convert to index: base year (earliest per state) = 100
income_indexed AS (
  SELECT
    geo_id,
    yr,
    avg_income_per_return,
    ROUND((avg_income_per_return / FIRST_VALUE(avg_income_per_return) OVER (PARTITION BY geo_id ORDER BY yr)) * 100, 1) AS income_index
  FROM income_raw
),
-- National income index (average across states)
national_income AS (
  SELECT yr, ROUND(AVG(income_index), 1) AS income_index
  FROM income_indexed
  GROUP BY 1
),
geo AS (
  SELECT geo_id, geo_name
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX
  WHERE level = 'State'
),
-- National rows: all metrics
national AS (
  SELECT
    c.month AS date,
    'country/USA' AS geo_id,
    'United States' AS geo_name,
    ROUND(c.cpi_index, 2) AS cpi_index,
    ROUND(((c.cpi_index - LAG(c.cpi_index, 12) OVER (ORDER BY c.month))
      / NULLIF(LAG(c.cpi_index, 12) OVER (ORDER BY c.month), 0)) * 100, 2) AS inflation_pct,
    m30.mortgage_rate_30yr_pct,
    m15.mortgage_rate_15yr_pct,
    nu.unemployment_rate_pct,
    ni.income_index
  FROM cpi c
  LEFT JOIN mortgage_30yr m30 ON c.month = m30.month
  LEFT JOIN mortgage_15yr m15 ON c.month = m15.month
  LEFT JOIN national_unemployment nu ON c.month = nu.month
  LEFT JOIN national_income ni ON YEAR(c.month) = ni.yr
),
-- State rows: unemployment + income (CPI and mortgage are national only)
states AS (
  SELECT
    u.month AS date,
    u.geo_id,
    g.geo_name,
    NULL::FLOAT AS cpi_index,
    NULL::FLOAT AS inflation_pct,
    NULL::FLOAT AS mortgage_rate_30yr_pct,
    NULL::FLOAT AS mortgage_rate_15yr_pct,
    u.unemployment_rate_pct,
    ii.income_index
  FROM unemployment u
  JOIN geo g ON u.geo_id = g.geo_id
  LEFT JOIN income_indexed ii ON u.geo_id = ii.geo_id AND YEAR(u.month) = ii.yr
)
SELECT * FROM national
UNION ALL
SELECT * FROM states
ORDER BY date, geo_id;

## Data Exploration

Our Iceberg table now has one row per month with all metrics pre-joined. After running the cell below:

1. Click the **Chart** tab to see inflation vs mortgage rates on a timeline
2. Try the **Pivot** button to compare years side-by-side

> This is exactly the kind of query Cortex Analyst will generate automatically from the semantic view.

In [ ]:
-- Compare CPI index growth vs income index growth (national)
-- If income_index < cpi_index, purchasing power is shrinking
SELECT
  date,
  cpi_index,
  income_index,
  inflation_pct,
  mortgage_rate_30yr_pct,
  unemployment_rate_pct
FROM hol_db.public.economic_indicators
WHERE geo_id = 'country/USA'
  AND date >= '2015-01-01'
  AND inflation_pct IS NOT NULL
ORDER BY date;

## Semantic View (Cortex Analyst)

**Cortex Analyst** uses a semantic view to understand your data's business meaning — dimensions, metrics, relationships. This is the bridge between raw tables and natural-language questions.

The cell below creates the semantic view via SQL using `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`. It defines:
- **Dimensions:** `DATE`, `GEO_ID`, `GEO_NAME`
- **Metrics:** `CPI_INDEX`, `INFLATION_PCT`, `MORTGAGE_RATE_30YR_PCT`, `MORTGAGE_RATE_15YR_PCT`, `UNEMPLOYMENT_RATE_PCT`, `INCOME_INDEX`

In [ ]:
-- Create the semantic view using YAML specification
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'hol_db.public',
  $$
name: economic_semantic_view
tables:
  - name: economic_indicators
    base_table:
      database: HOL_DB
      schema: PUBLIC
      table: ECONOMIC_INDICATORS
    dimensions:
      - name: DATE
        description: "Date of the observation"
        expr: economic_indicators.DATE
        data_type: DATE
      - name: GEO_ID
        description: "Geographic area identifier"
        expr: economic_indicators.GEO_ID
        data_type: TEXT
      - name: GEO_NAME
        description: "Geographic area — United States for national, or state name (e.g. California)"
        expr: economic_indicators.GEO_NAME
        data_type: TEXT
    facts:
      - name: CPI_INDEX
        description: "Consumer Price Index, base period 1982-84 = 100 (national only)"
        expr: economic_indicators.CPI_INDEX
        data_type: NUMBER
      - name: INFLATION_PCT
        description: "Year-over-year inflation rate as percent (national only)"
        expr: economic_indicators.INFLATION_PCT
        data_type: NUMBER
      - name: MORTGAGE_RATE_30YR_PCT
        description: "30-year fixed mortgage rate, national average, percent (national only)"
        expr: economic_indicators.MORTGAGE_RATE_30YR_PCT
        data_type: NUMBER
      - name: MORTGAGE_RATE_15YR_PCT
        description: "15-year fixed mortgage rate, national average, percent (national only)"
        expr: economic_indicators.MORTGAGE_RATE_15YR_PCT
        data_type: NUMBER
      - name: UNEMPLOYMENT_RATE_PCT
        description: "Unemployment rate as percent (available national and by state)"
        expr: economic_indicators.UNEMPLOYMENT_RATE_PCT
        data_type: NUMBER
      - name: INCOME_INDEX
        description: "Average income per tax return, indexed to earliest available year = 100. Compare to CPI_INDEX to assess purchasing power. (available national and by state, annual grain)"
        expr: economic_indicators.INCOME_INDEX
        data_type: NUMBER
    metrics:
      - name: AVG_CPI_INDEX
        description: "Average Consumer Price Index"
        expr: AVG(economic_indicators.CPI_INDEX)
      - name: AVG_INFLATION_PCT
        description: "Average year-over-year inflation rate"
        expr: AVG(economic_indicators.INFLATION_PCT)
      - name: AVG_MORTGAGE_RATE_30YR
        description: "Average 30-year fixed mortgage rate"
        expr: AVG(economic_indicators.MORTGAGE_RATE_30YR_PCT)
      - name: AVG_UNEMPLOYMENT_RATE
        description: "Average unemployment rate"
        expr: AVG(economic_indicators.UNEMPLOYMENT_RATE_PCT)
      - name: AVG_INCOME_INDEX
        description: "Average income index"
        expr: AVG(economic_indicators.INCOME_INDEX)
$$
);

In [ ]:
%%sql -r semantic_views
-- Confirm the semantic view exists
SHOW SEMANTIC VIEWS IN SCHEMA hol_db.public;

## Cortex Agent

A **Cortex Agent** wraps your semantic view into a conversational interface. It translates natural-language questions into SQL via Cortex Analyst, executes them, and returns grounded answers.

You can also create agents from the UI: go to **AI & ML → Cortex Agents** to see the visual builder, configure tools, and test interactively. We'll use SQL here for reproducibility.

In [ ]:
-- Create a Cortex Agent backed by the economic indicators semantic view
CREATE OR REPLACE AGENT hol_db.public.hol_economic_agent
  FROM SPECIFICATION $$
  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: economic_analyst
        description: "Answers questions about US economic indicators: CPI/inflation, mortgage interest rates (30-year, 15-year), unemployment rate (national and by state), and income index."

  tool_resources:
    economic_analyst:
      semantic_view: HOL_DB.PUBLIC.ECONOMIC_SEMANTIC_VIEW
      execution_environment:
        type: warehouse
        warehouse: HOL_WH
  $$;

In [ ]:
-- Verify the agent is created
SHOW AGENTS IN SCHEMA hol_db.public;

-- Grant consumer role usage on the agent
GRANT USAGE ON AGENT hol_db.public.hol_economic_agent TO ROLE end_user_role;
GRANT SELECT ON SEMANTIC VIEW hol_db.public.economic_semantic_view TO ROLE end_user_role;
GRANT SELECT ON TABLE hol_db.public.economic_indicators TO ROLE end_user_role;

## CoWork

CoWork provides a chat interface for your agents — ideal for business users.

1. Go to **AI & ML → Snowflake Intelligence** in Snowsight
2. Make sure you're using role `end_user_role` and warehouse `hol_wh`
3. Select your **hol_economic_agent**
4. Ask a question:
   ```
   How has the 30-year mortgage rate changed relative to inflation since 2020?
   ```
5. Review the grounded answer — note you can inspect the generated SQL

> Same agent, same semantic view, same answer — just a different (consumer) role and surface.

## MCP Server & OAuth

We expose the agent externally via a **Snowflake-managed MCP server** + OAuth. This lets any MCP-compatible client (like Gemini Enterprise) call your agent over a secure connection.

In [ ]:
-- MCP server exposing the Cortex Agent as a tool
CREATE OR REPLACE MCP SERVER hol_db.public.hol_mcp
  FROM SPECIFICATION $$
  tools:
    - name: "hol-economic-agent"
      type: "CORTEX_AGENT_RUN"
      identifier: "HOL_DB.PUBLIC.HOL_ECONOMIC_AGENT"
      description: "US economic indicators agent — answers questions about inflation (CPI), mortgage rates, unemployment, and income."
      title: "Economic Indicators Agent"
  $$;

In [ ]:
-- OAuth security integration for external MCP clients
CREATE OR REPLACE SECURITY INTEGRATION hol_mcp_oauth
  TYPE = OAUTH
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://vertexaisearch.cloud.google.com/oauth-redirect'
  ENABLED = TRUE;

-- Retrieve all credentials needed for Gemini Enterprise MCP connection
WITH secrets AS (
  SELECT PARSE_JSON(SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('HOL_MCP_OAUTH')) AS s
),
account_url AS (
  SELECT 'https://' || CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() || '.snowflakecomputing.com' AS base
)
SELECT field_name, value
FROM (
  SELECT 1 AS ord, 'MCP Server URL'   AS field_name, a.base || '/api/v2/cortex/mcp' AS value FROM account_url a
  UNION ALL
  SELECT 2, 'Auth URL',              a.base || '/oauth/authorize' FROM account_url a
  UNION ALL
  SELECT 3, 'Auth URL Params',       'response_type=code&code_challenge_method=S256' FROM account_url a
  UNION ALL
  SELECT 4, 'Token URL',             a.base || '/oauth/token-request' FROM account_url a
  UNION ALL
  SELECT 5, 'Client ID',             s.s:OAUTH_CLIENT_ID::STRING FROM secrets s
  UNION ALL
  SELECT 6, 'Client Secret',         s.s:OAUTH_CLIENT_SECRET::STRING FROM secrets s
  UNION ALL
  SELECT 7, 'Scopes',                'session:role:end_user_role' FROM secrets s
  UNION ALL
  SELECT 8, 'MCP Server Description', 'Snowflake Cortex Agent for US economic indicators (CPI, mortgage rates, unemployment, income)' FROM secrets s
  UNION ALL
  SELECT 9, 'Agent Instructions',    'Use the hol-economic-agent tool to answer questions about US economic data including inflation, mortgage rates, unemployment by state, and income trends.' FROM secrets s
  UNION ALL
  SELECT 10, 'Data Connector Name',  'hol_cortex_gemini_economic_agent' FROM secrets s
  UNION ALL
  SELECT 11, 'Data Policy',          '(optional) projects/<PROJECT_ID>/locations/<LOCATION>/contentPolicies/<POLICY_NAME>' FROM secrets s
)
ORDER BY ord;

## Gemini Enterprise — Register MCP

In the **Google Cloud Console**, register the Snowflake MCP server as a data store:

1. Go to **Vertex AI → Agent Builder → Data Stores → Create data store**
2. Select **Custom MCP Server**
3. Fill in the fields using the values from the table output above
4. Complete the OAuth login flow when prompted
5. **Enable Actions** — your Cortex Agent tools appear as available actions

## Gemini Enterprise — Query

In the **Gemini Enterprise** chat interface, ask:

```
How has the 30-year mortgage rate changed relative to inflation since 2020?
```

Gemini routes the request through MCP → OAuth → Snowflake MCP Server → Cortex Agent → Semantic View → Iceberg Table and returns a grounded answer.

> Same agent. Same data. Different surface. No copies.

### Troubleshooting

**Network Policy** — If Gemini Enterprise cannot reach your Snowflake account (OAuth errors, timeouts), your account may have a network policy blocking external IPs. Run the cell below to temporarily allow all connections.

**GCP Org Policy: Custom MCP Server blocked** — If you see `constraints/discoveryengine.managed.disableCustomMcpServerConnector` after clicking Create:
1. Go to **Google Cloud Console → IAM & Admin → Organization Policies**
2. Search for `disableCustomMcpServerConnector`
3. Click the constraint → **Manage Policy** → set **Enforcement** to **Off** for your project
4. Save and retry the data connector creation

In [ ]:
%%sql -r network_policy_result
-- Temporarily disable account network policy to allow Gemini Enterprise OAuth
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT UNSET NETWORK_POLICY;

-- To re-enable later:
-- ALTER ACCOUNT SET NETWORK_POLICY = <your_policy_name>;

## Done!

You built **one governed data product** and consumed it from **three surfaces**:

| Surface | How |
|---------|-----|
| **CoCo** | Auto-discovers semantic view, answers data questions directly |
| **CoWork** | Business user chat interface (native) |
| **Gemini Enterprise** | External AI via MCP + OAuth |

All powered by:
- **Iceberg** open table format on GCS
- **Semantic View** providing business context
- **Cortex Agent** translating questions to grounded SQL
- **MCP Server** exposing the agent securely to external clients

## Cleanup

Run this cell to tear down **all** Snowflake objects created during the workshop. This returns your account to a blank state.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Drop database (cascades all objects inside: tables, views, agents, MCP servers)
DROP DATABASE IF EXISTS hol_db;
DROP WAREHOUSE IF EXISTS hol_wh;
DROP INTEGRATION IF EXISTS hol_mcp_oauth;
DROP ROLE IF EXISTS hol_role;
DROP ROLE IF EXISTS end_user_role;

-- NOTE: hol_gcs_vol is kept — GCS bucket permissions take time to set up
-- To drop it manually: DROP EXTERNAL VOLUME IF EXISTS hol_gcs_vol;

-- Re-enable network policy if it was disabled
-- ALTER ACCOUNT SET NETWORK_POLICY = ACCOUNT_VPN_POLICY_SE;

SHOW ROLES LIKE '%HOL%';